# Lie Symmetry Analysis of the 1D Linear Heat Equation

This tutorial demonstrates using **`symlie`** to analyze the classical linear heat conduction equation:
$$u_t - u_{xx} = 0$$

We will compute:
1. The Fréchet linearization and formal adjoint operators.
2. The determining system of linear PDEs for the infinitesimals.
3. Point symmetries within a finite polynomial ansatz, followed by explicit Galilean and projective generators.
4. The Lie bracket commutator algebra.
5. Similarity reduction to the self-similar Gaussian fundamental solution.

In [ ]:
import sympy as sp

from symlie import (
    InfinitesimalGenerator,
    adjoint_frechet_derivative,
    determining_equations,
    frechet_derivative,
    infinitesimals,
    lie_bracket,
    max_derivative_order,
    verify_generator,
)

sp.init_printing()

x, t = sp.symbols("x t")
u = sp.Function("u")(x, t)

# Heat equation
heat_eq = u.diff(t) - u.diff(x, 2)
print("Derivative Order:", max_derivative_order(heat_eq, u, (x, t)))
sp.Eq(heat_eq, 0)

## 1. Linearization and Formal Adjoint

For the linear heat equation, the forward Fréchet derivative is $D(Q) = Q_t - Q_{xx}$, and its formal adjoint under integration by parts is $D^*(v) = -v_t - v_{xx}$.

In [ ]:
Q = sp.Function("Q")(x, t)
v = sp.Function("v")(x, t)

D_heat = frechet_derivative(heat_eq, u, (x, t), Q)
D_star = adjoint_frechet_derivative(heat_eq, u, (x, t), v)

print("Fréchet Derivative D(Q):")
display(D_heat)
print("Formal Adjoint D*(v):")
display(D_star[0])

## 2. Determining System of Linear PDEs

We construct the determining equations for point symmetries $\xi^x(x, t, u), \xi^t(x, t, u), \phi^u(x, t, u)$:

In [ ]:
det_sys = determining_equations(heat_eq, u, (x, t))
print(f"Total determining equations: {len(det_sys.equations)}")
for eq in det_sys.equations[:6]:
    display(eq)

## 3. Polynomial-Ansatz Lie Point Symmetries

`infinitesimals(ansatz_degree=1)` searches only within total-degree-1 polynomials in $(x,t,u)$. The resulting six-dimensional space is not the complete heat-equation symmetry algebra: it contains the solution shifts $\partial_u$ and $x\partial_u$, while the Galilean and projective generators require higher polynomial degrees. More generally, every heat-equation solution $h(x,t)$ gives the symmetry $h\partial_u$, so the full algebra contains an infinite-dimensional ideal.

In [ ]:
sol = infinitesimals(heat_eq, u, (x, t), ansatz_degree=1)
print(f"Dimension within degree-1 ansatz: {sol.ansatz_dimension}\n")

for i, gen in enumerate(sol.basis, 1):
    is_valid = verify_generator(heat_eq, u, (x, t), gen)
    print(f"X_{i}: xi^x = {gen.xi[0]},  xi^t = {gen.xi[1]},  phi^u = {gen.phi[0]}")
    print(f"      Verified invariant: {is_valid}")

print("\nGeneral Linear Combination:")
display(sol.general)

X_galilean = InfinitesimalGenerator(xi=(2 * t, 0), phi=(-x * u,))
X_projective = InfinitesimalGenerator(
    xi=(4 * t * x, 4 * t**2),
    phi=(-(x**2 + 2 * t) * u,),
)
assert verify_generator(heat_eq, u, (x, t), X_galilean)
assert verify_generator(heat_eq, u, (x, t), X_projective)
print("Galilean and projective generators verified separately.")

## 4. Commutator Lie Algebra

We calculate the Lie bracket $[X_1, X_3]$ between space translation $X_1 = \partial_x$ and parabolic scaling $X_3 = x \partial_x + 2t \partial_t$:

In [ ]:
X1 = sol.basis[0]  # d/dx
X3 = sol.basis[2]  # x d/dx + 2t d/dt

bracket = lie_bracket(X1, X3, u, (x, t))
print("[X_1, X_3] =", bracket)
assert bracket.xi == (1, 0) and bracket.phi == (0,)
print("Commutator verified: [X_1, X_3] = X_1")

## 5. Self-Similar Reduction: The Gaussian Heat Kernel

Using the parabolic scaling generator $X = x \partial_x + 2t \partial_t - u \partial_u$:
The similarity variable is $\xi = \frac{x}{\sqrt{t}}$, and $u(x, t) = \frac{1}{\sqrt{t}} F(\xi)$.

Substituting yields the Gaussian fundamental solution (heat kernel):
$$u(x, t) = \frac{1}{\sqrt{4\pi t}} e^{-\frac{x^2}{4t}}$$

In [ ]:
gaussian_solution = (1 / sp.sqrt(4 * sp.pi * t)) * sp.exp(-(x**2) / (4 * t))
print("Gaussian Fundamental Solution:")
display(gaussian_solution)

# Verify that it satisfies the heat equation identically
residual = sp.simplify(heat_eq.subs(u, gaussian_solution).doit())
print("Residual in Heat Equation:", residual)
assert residual == 0
print("Verification: The Gaussian solution satisfies u_t - u_xx = 0!")